In [ ]:
# Import packages
import io
from datetime import date, timedelta

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sbn
from IPython.display import Markdown, display


In [ ]:
###################
#SET E-MAIL HEADER#
###################

# This cell will serve as a header in your email. You can add some information here that may be useful for providing context. 
# If you want to change the actual text that appears, feel free to edit the "md_text" variable directly.

# Input a title of your choosing here.
title = "Weekly Report | Cleveland Public Health"

# Write a brief description about the analysis.
description = """
A weekly email that highlights:
* New public heath complaints from the last week
* Complaints resolved within the last week
* Unresolved complaints older than 30 days
* A comparison of new complaints verses resolved complaints over the last 30 days
"""

# Get today's date
current_date = date.today()

# Print markdown header
md_text = f"""
## {title}
**Description**
{description}
**Data as of**  
{current_date}
"""

# Render it in the output
display(Markdown(md_text))

In [ ]:
###########
#PULL DATA#
###########

# Put the URL for your API request here. You can do this using the query builder in ArcGIS Online.
url = "https://services3.arcgis.com/dty2kHktVXHrqO8i/arcgis/rest/services/CDPH_Complaints/FeatureServer/0/query"

# Type out a where clause here. 
# You can utilize an "f string" to make this filter dynamic.
# NOTE: For many feature layers, the maximum amount of records the ArcGIS Online API can query is 2,000. You'll need to perform multiple queries if you are reading in more than 2k records.
days_ago_30 = str(current_date - timedelta(days=30))
where_clause = f"""
submit_date >= '{days_ago_30}'
"""

# Set query parameters. Nothing here for you to do.
query_params = {
    "where": where_clause,       # The where clause from above.
    "returnGeometry": "false",    # We're not doing any work with spatial data. But if you want to make maps with your data, set to 'true'.
    "f": "json"                  # Tells the server to respond with JSON format.
}

## SUBMIT AND PARSE API REQUEST(S)
# Since in many cases we are limited to reading 2,000 records per query, this function will make several API requests to get all the records.
# It will also parse the request and extract the data into a list of records.
def get_data(url, query_params):
    """
    This function retrieves data from ArcGIS Online FeatureLayer via a series of GET requests. 
    It will pull data in groups of 2,000 records, and append all the data to one list of records.

    Args:
        url (str): Spark session context.
        query_params (dict): A spark DataFrame to geocode.

    Returns:
        list: The spark DataFrame, with new geocoded columns appended.
    """
    # Get record count
    record_count = requests.get(url,params={"where":query_params['where'],"returnCountOnly":"true","f":"json"}).json()['count']

    # Split record count into offsets
    offsets = range(0,record_count,1000)
    
    # List of data records
    results = []
    
    # Loop through offsets and get data for each offset
    for i, offset in enumerate(offsets):
        # Perform a GET request with the given offset
        query_params['resultOffset'] = offset
        req = requests.get(url,params=query_params)
        # Get JSON
        resp = req.json()

        # Extract data and append it to our final result
        data = [a['attributes'] for a in resp['features']]

        results += data
        
    # Ensure the number of records matches the record count of the Feature Layer.
    assert len(results) == record_count

    return results

data = get_data(url, query_params)



In [22]:
######################
#CONVERT TO DATAFRAME#
######################
# If you use the get_data function from above, your data should look something like this:.
"""
[{'service_request_id': '202000403109',
  'service_category': 'Trash & Recycling',
  'service_name': 'Waste Cart Concerns'},
 {'service_request_id': '202000403083',
  'service_category': 'Building & Housing',
  'service_name': 'Electrical Issue'},
 {'service_request_id': '202000403082',
  'service_category': 'Street Issues',
  'service_name': 'Debris in Street'}]
"""

# The format above is known as "records" format, and will allow you to automatically convert to a pandas dataframe when doing pd.DataFrame(records).
# Try converting to DataFrame below:
pd.DataFrame(data)

,ObjectId,id,complaint_number,submit_date,submit_time,complaint_type,complaint_input,complaint_inspector,complaint_status,complaint_outcome,...,permanent_parcel_number,census_tract,ward_number,submit_datetime,dw_ward,dw_ward_2014,dw_ward_2026,dw_neighborhood,dw_census_tract,dw_parcel
0,38001,42343,2026042343,1787270400000,12:46PM,High Grass/Weeds,Website Intake,Robert Peterson,Active,NaN,...,NaN,NaN,11,1787330760000,12.0,11.0,12.0,Jefferson,39035124100,01804060
1,38003,42344,2026042344,1787270400000,12:49PM,Insect/Vermin Infestation,Website Intake,Theyesa Bryant,Resolved,No Code Violation Observed,...,NaN,NaN,10,1787330940000,10.0,8.0,10.0,North Shore Collinwood,39035117201,11301003
2,38006,42345,2026042345,1787270400000,12:51PM,Food Complaints,Website Intake,Howard Jackson,Inspected,Consultation,...,NaN,NaN,5,1787331060000,5.0,5.0,5.0,Central,39035198400,12405001
3,38460,42346,2026042346,1787270400000,2:33PM,Food Complaints,Website Intake,Amiya Weaver,Active,Consultation,...,NaN,NaN,14,1787337180000,13.0,11.0,13.0,Bellaire-Puritas,39035196400,01921003
4,38463,42347,2026042347,1787270400000,2:39PM,Mold,Website Intake,Candice Houi-Foster,Active,NaN,...,NaN,NaN,5,1787337540000,5.0,5.0,5.0,Kinsman,39035114800,12515003
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
596,39803,42338,2026042338,1787270400000,11:38AM,High Grass/Weeds,Website Intake,Theyesa Bryant,Active,Ticketed,...,NaN,NaN,10,1787326680000,10.0,8.0,10.0,North Shore Collinwood,39035117600,11424051
597,39804,42339,2026042339,1787270400000,11:49AM,Accumulation of Garbage,Website Intake,Theyesa Bryant,Active,NaN,...,NaN,NaN,10,1787327340000,10.0,8.0,10.0,North Shore Collinwood,39035117600,11424051
598,39805,42340,2026042340,1787270400000,12:00PM,Accumulation of Garbage,Website Intake,Theyesa Bryant,Resolved,Ticketed,...,NaN,NaN,10,1787328000000,10.0,10.0,10.0,Euclid-Green,39035117900,11713039
599,39806,42341,2026042341,1787270400000,12:38PM,Accumulation of Garbage,311,Robert Peterson,Active,NaN,...,01314096,NaN,11,1787330280000,11.0,13.0,11.0,Old Brooklyn,39035106100,01314096


In [0]:
##########
#ANALYSIS#
##########

# Conduct your data analysis below. You should use the dataframe from above as your starting point. Feel free to add more cells to separate output.
# We import the "seaborn" package in the first cell above. You can use this or another package of your choosing for creating visualizations.
# If you choose to import additional packages, be sure to update the dependencies in your GitHub Action! Otherwise the workflow will fail.